In [26]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [27]:
df = pd.read_csv("diabetic_data.csv")
df.columns

Index(['encounter_id', 'patient_nbr', 'race', 'gender', 'age', 'weight',
       'admission_type_id', 'discharge_disposition_id', 'admission_source_id',
       'time_in_hospital', 'payer_code', 'medical_specialty',
       'num_lab_procedures', 'num_procedures', 'num_medications',
       'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1',
       'diag_2', 'diag_3', 'number_diagnoses', 'max_glu_serum', 'A1Cresult',
       'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
       'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide',
       'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone',
       'tolazamide', 'examide', 'citoglipton', 'insulin',
       'glyburide-metformin', 'glipizide-metformin',
       'glimepiride-pioglitazone', 'metformin-rosiglitazone',
       'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted'],
      dtype='str')

Data Cleaning/Preparation

Removed 'encounter_id', 'patient_nbr', and 'payer_code' from the dataset, given that our objective for this project is to identify which variables are more strongly related to hospital readmissions.

In [28]:
df_clean = df.copy()
df_clean = df_clean.drop(columns=[
    'encounter_id', 'patient_nbr','payer_code'])
df_clean.columns

Index(['race', 'gender', 'age', 'weight', 'admission_type_id',
       'discharge_disposition_id', 'admission_source_id', 'time_in_hospital',
       'medical_specialty', 'num_lab_procedures', 'num_procedures',
       'num_medications', 'number_outpatient', 'number_emergency',
       'number_inpatient', 'diag_1', 'diag_2', 'diag_3', 'number_diagnoses',
       'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide',
       'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide',
       'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose',
       'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton',
       'insulin', 'glyburide-metformin', 'glipizide-metformin',
       'glimepiride-pioglitazone', 'metformin-rosiglitazone',
       'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted'],
      dtype='str')

In [29]:

# to see how many rows in the dataset has

num_row = df_clean.shape[0]
print(num_row)

101766


In [30]:
# to see how many ? are in each column
sum_q = (df_clean == '?').sum()
amount_q =sum_q[sum_q>0]
print(num_row - amount_q)

race                  99493
weight                 3197
medical_specialty     51817
diag_1               101745
diag_2               101408
diag_3               100343
dtype: int64


In [31]:
print("Amount Missing : ", sum_q['weight'])
print("Amount Available: ", (num_row - amount_q['weight']) )
print("Perentage Missing:" , round((sum_q['weight']/num_row)*100,2))

Amount Missing :  98569
Amount Available:  3197
Perentage Missing: 96.86


Weight can be a reasonable candidate for removal

In [32]:
# check to see which have yes and no 
temp = []
for column in df_clean.columns:
    if 'Yes' in df_clean[column].values or 'No' in df_clean[column].values:
        temp.append(column)
print(temp)

['metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed']


In [33]:
df_clean[temp].apply(lambda x:x.value_counts())

,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,...,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed
Ch,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,47011.0,NaN
Down,575.0,45.0,11.0,1.0,194.0,NaN,560.0,564.0,NaN,118.0,...,NaN,NaN,12218.0,6.0,NaN,NaN,NaN,NaN,NaN,NaN
No,81778.0,100227.0,101063.0,101680.0,96575.0,101765.0,89080.0,91116.0,101743.0,94438.0,...,101766.0,101766.0,47383.0,101060.0,101753.0,101765.0,101764.0,101765.0,54755.0,23403.0
Steady,18346.0,1384.0,668.0,79.0,4670.0,1.0,11356.0,9274.0,23.0,6976.0,...,NaN,NaN,30849.0,692.0,13.0,1.0,2.0,1.0,NaN,NaN
Up,1067.0,110.0,24.0,6.0,327.0,NaN,770.0,812.0,NaN,234.0,...,NaN,NaN,11316.0,8.0,NaN,NaN,NaN,NaN,NaN,NaN
Yes,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,78363.0


Looking at the dataset, examide and citoglipton both have only "No" for all 101,766 observations, showing that these variables aren't good candidates for identifying a relationship with hospital readmission.

In [34]:
df_clean =df_clean.drop(columns=['examide','citoglipton'])

To get a better look at the variables acetohexamide, tolbutamide, glipizide-metformin, glimepiride-pioglitazone,	metformin-rosiglitazone, metformin-pioglitazone, change, and diabetesMed, I examined their value counts.

In [35]:
l = ['acetohexamide', 'tolbutamide', 'glipizide-metformin', 'glimepiride-pioglitazone','metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed']
df_clean[l].apply(lambda x:x.value_counts())



,acetohexamide,tolbutamide,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed
Ch,NaN,NaN,NaN,NaN,NaN,NaN,47011.0,NaN
No,101765.0,101743.0,101753.0,101765.0,101764.0,101765.0,54755.0,23403.0
Steady,1.0,23.0,13.0,1.0,2.0,1.0,NaN,NaN
Yes,NaN,NaN,NaN,NaN,NaN,NaN,NaN,78363.0


Looking at the dataset, acetohexamide, glimepiride-pioglitazone, and metformin-pioglitazone have only one person who is "Steady," meaning the patient took their medicine and the dosage did not change. These variables are strong candidates for removal because they give us almost no variation when investigating the relationship with hospital readmission. For this reason, they will be taken out of the dataset.

In [36]:
print(df_clean.columns.tolist())

['race', 'gender', 'age', 'weight', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'medical_specialty', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1', 'diag_2', 'diag_3', 'number_diagnoses', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted']


In [37]:
df_clean =df_clean.drop(columns=['acetohexamide', 'glimepiride-pioglitazone', 'metformin-pioglitazone'])


In [41]:
temp = [x for x in temp if x not in['acetohexamide', 'glimepiride-pioglitazone', 'metformin-pioglitazone','examide','citoglipton']]
df_clean[temp].apply(lambda x:x.value_counts())

,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,insulin,glyburide-metformin,glipizide-metformin,metformin-rosiglitazone,change,diabetesMed
Ch,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,47011.0,NaN
Down,575.0,45.0,11.0,1.0,194.0,560.0,564.0,NaN,118.0,87.0,3.0,5.0,NaN,NaN,12218.0,6.0,NaN,NaN,NaN,NaN
No,81778.0,100227.0,101063.0,101680.0,96575.0,89080.0,91116.0,101743.0,94438.0,95401.0,101458.0,101728.0,101763.0,101727.0,47383.0,101060.0,101753.0,101764.0,54755.0,23403.0
Steady,18346.0,1384.0,668.0,79.0,4670.0,11356.0,9274.0,23.0,6976.0,6100.0,295.0,31.0,3.0,38.0,30849.0,692.0,13.0,2.0,NaN,NaN
Up,1067.0,110.0,24.0,6.0,327.0,770.0,812.0,NaN,234.0,178.0,10.0,2.0,NaN,1.0,11316.0,8.0,NaN,NaN,NaN,NaN
Yes,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,78363.0


To get a better look at the variables tolbutamide, troglitazone, glipizide-metformin, and metformin-rosiglitazone, I examined their value counts.	

In [44]:
med_list = ['tolbutamide', 'troglitazone', 'glipizide-metformin', 'metformin-rosiglitazone'	]
df_clean[med_list].apply(lambda x:x.value_counts())

,tolbutamide,troglitazone,glipizide-metformin,metformin-rosiglitazone
No,101743,101763,101753,101764
Steady,23,3,13,2


Looking at the data for tolbutamide, troglitazone, glipizide-metformin,and metformin-rosiglitazone,the results are same to the previous variables. "Steady" has few observations compared to "No". This shows that these are good candidates for removal since they have little variation.

In [45]:
df_clean =df_clean.drop(columns=['tolbutamide', 'troglitazone', 'glipizide-metformin', 'metformin-rosiglitazone'])
